# ContraTICO — Baseline Pipeline

Full end-to-end pipeline for the ContraTICO baseline:
1. **QG** — Question Generation (calls `QG/code/{model}.py` with prompts from `QG/code/prompt.py`)
2. **QA** — Question Answering source + BT (calls `QA/code/{model}.py` with prompt from `QA/code/prompt.py`)
3. **Mapping** — Map QG / QA source / QA BT
4. **String Comparison** — F1, EM, chrF, BLEU
5. **SBERT** — Sentence-BERT cosine similarity

| Parameter | Values |
|-----------|--------|
| Model | Qwen/Qwen2.5-3B-Instruct (swap script for a different model) |
| Prompts | vanilla, atomic, semantic |
| Languages | es, fr, hi, tl, zh |
| Perturbations | alteration, omission |

## 0. Environment Setup

In [ ]:
import os, sys, subprocess, json

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')
print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CACHE_DIR = '/content/drive/MyDrive/AskQE_Models_Cache'
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME'] = DRIVE_CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = os.path.join(DRIVE_CACHE_DIR, 'transformers')
    os.environ['SENTENCE_TRANSFORMERS_HOME'] = os.path.join(DRIVE_CACHE_DIR, 'sentence_transformers')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'torch', 'accelerate', 'nltk',
                'sentence-transformers', 'sacrebleu'], check=True)
print('Dependencies installed!')

In [ ]:
if IN_KAGGLE:
    PROJECT_ROOT = '/kaggle/working/askqe'
elif IN_COLAB:
    PROJECT_ROOT = '/content/askqe'
else:
    PROJECT_ROOT = os.getcwd()

if not os.path.exists(PROJECT_ROOT) and (IN_KAGGLE or IN_COLAB):
    subprocess.run(['git', 'clone',
                    'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git',
                    PROJECT_ROOT], check=True)

print(f'Project root: {PROJECT_ROOT}')

## 1. Configuration

To use a different model, change `MODEL_SHORT` to point to the model-specific scripts.

In [ ]:
# ===================================================
# CHANGE THESE TO USE A DIFFERENT MODEL
# Available: qwen-3b, gemma-9b, gemma-27b, llama-8b, llama-70b, yi-9b
# ===================================================
MODEL_SHORT = 'qwen-3b'
QG_SCRIPT   = f'{PROJECT_ROOT}/QG/code/{MODEL_SHORT}.py'
QA_SCRIPT   = f'{PROJECT_ROOT}/QA/code/{MODEL_SHORT}.py'

BASELINE_DIR = f'{PROJECT_ROOT}/Qwen2.5-3B-Instruct/contratico/baseline'
EVAL_DIR     = f'{BASELINE_DIR}/evaluation'

PROMPTS       = ['vanilla', 'atomic', 'semantic']
LANGUAGES     = ['es', 'fr', 'hi', 'tl', 'zh']
PERTURBATIONS = ['alteration', 'omission']

# Create output directories
for d in ['QG', 'QA/source', 'QA/bt', 'mapping',
          'evaluation/sbert', 'evaluation/string comparison']:
    os.makedirs(f'{BASELINE_DIR}/{d}', exist_ok=True)

print(f'Model: {MODEL_SHORT}')
print(f'QG script: {QG_SCRIPT}')
print(f'QA script: {QA_SCRIPT}')

## 2. Pre-download Models

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import torch

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
print(f'[1/2] Caching {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
del model, tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('      Done')

print('[2/2] Caching SBERT...')
_ = SentenceTransformer('all-MiniLM-L6-v2')
del _
print('      Done')

## 3. QG — Question Generation

Calls `QG/code/{model}.py` for each prompt strategy (vanilla, atomic, semantic).
Prompts are defined in `QG/code/prompt.py`.

In [ ]:
for prompt in PROMPTS:
    qg_output = f'{BASELINE_DIR}/QG/{prompt}_{MODEL_SHORT}.jsonl'

    cmd = [
        sys.executable, '-u', QG_SCRIPT,
        '--output_path', qg_output,
        '--prompt', prompt
    ]

    print(f'Running QG [{prompt}]...')
    subprocess.run(cmd, check=True)
    print(f'Done: QG [{prompt}] -> {qg_output}')

print('\nAll QG complete!')

## 4. QA — Question Answering

Calls `QA/code/{model}.py` with `--run_all` to process all configs.
QA prompt defined in `QA/code/prompt.py`.

In [ ]:
cmd = [
    sys.executable, '-u', QA_SCRIPT,
    '--run_all'
]

print('Running QA (all configs: source + BT x languages x perturbations x pipelines)...')
subprocess.run(cmd, check=True)
print('All QA complete!')

## 5. Mapping

In [ ]:
mapping_script = f'{BASELINE_DIR}/mapping/mapping_contratico.py'
cmd = [sys.executable, '-u', mapping_script,
       '--base_dir', BASELINE_DIR]
print('Running mapping...')
subprocess.run(cmd, check=True)
print('Mapping complete!')

## 6. String Comparison

In [ ]:
str_script = f'{EVAL_DIR}/string comparison/string_comparison_contratico.py'
cmd = [sys.executable, '-u', str_script,
       '--base_dir', BASELINE_DIR]
print('Running string comparison...')
subprocess.run(cmd, check=True)
print('String comparison complete!')

## 7. SBERT Evaluation

In [ ]:
sbert_script = f'{EVAL_DIR}/sbert/sbert_contratico.py'
cmd = [sys.executable, '-u', sbert_script,
       '--base_dir', BASELINE_DIR]
print('Running SBERT evaluation...')
subprocess.run(cmd, check=True)
print('SBERT evaluation complete!')

## Summary

Full ContraTICO baseline pipeline complete! Output structure:
```
baseline/
  QG/{prompt}_{model}.jsonl
  QA/source/en-{prompt}.jsonl
  QA/bt/{lang}-{prompt}-{perturbation}.jsonl
  mapping/{lang}-{pipeline}-{pert}.jsonl
  evaluation/
    sbert/{lang}-{pipeline}.jsonl
    string comparison/{lang}-{pipeline}.jsonl
```